In [1]:
!nvidia-smi

Sat Jun  6 06:24:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.124.06             Driver Version: 570.124.06     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA RTX 6000 Ada Gene...    On  |   00000000:E1:00.0 Off |                  Off |
| 30%   26C    P8             21W /  300W |       2MiB /  49140MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers==4.40.1 tokenizers==0.19.1 timm==0.9.10 accelerate


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip


In [3]:
import transformers, timm
print("transformers:", transformers.__version__)  # 要 4.40.1
print("timm:", timm.__version__)                    # 要 0.9.10

transformers: 4.40.1
timm: 0.9.10


In [4]:
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch

processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)

model = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,   # 半精度,~14GB,48GB 卡轻松装下
    low_cpu_mem_usage=True,
).to("cuda")                       # 这次可以正常 .to(),因为没量化

print("模型设备:", next(model.parameters()).device)
print("模型加载完成 ✅")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


processor_config.json:   0%|          | 0.00/130 [00:00<?, ?B/s]

processing_prismatic.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/openvla/openvla-7b:
- processing_prismatic.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


preprocessor_config.json: 0.00B [00:00, ?B/s]

...

model-00002-of-00003.safetensors:   0%|          | 0.00/6.97G [00:00<?, ?B/s]

model-00003-of-00003.safetensors:   0%|          | 0.00/1.16G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/136 [00:00<?, ?B/s]

模型设备: cuda:0
模型加载完成 ✅


In [6]:
from PIL import Image
import requests

# 拿一张示例机械臂场景图(OpenVLA 训练分布内的视角)
url = "https://raw.githubusercontent.com/openvla/openvla/main/prismatic/vla/datasets/rlds/oxe/materials/widowx.jpg"
try:
    image = Image.open(requests.get(url, stream=True).raw).convert("RGB")
except Exception:
    # 备用:用一张随机彩色图占位,先验证推理管线能跑通
    import numpy as np
    image = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))

# 语言指令——OpenVLA 要求这个固定格式
instruction = "pick up the object"
prompt = f"In: What action should the robot take to {instruction}?\nOut:"

# 重新构造 inputs,确保 input_ids / pixel_values 是正确的 tensor
inputs = processor(prompt, image)
inputs = {k: v.to("cuda", dtype=torch.bfloat16) if v.dtype == torch.float32 else v.to("cuda")
          for k, v in inputs.items()}

action = model.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)

print("指令:", instruction)
print("输出动作向量 (7维):", action)
print("含义: [Δx, Δy, Δz, Δroll, Δpitch, Δyaw, gripper]")

指令: pick up the object
输出动作向量 (7维): [-0.00289288  0.02482914  0.00443561  0.00880144 -0.0121683   0.06601701
  0.        ]
含义: [Δx, Δy, Δz, Δroll, Δpitch, Δyaw, gripper]


In [7]:
# 装系统级的离屏渲染依赖(MuJoCo 在无头服务器上需要)
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libosmesa6 libglfw3 libglew-dev patchelf > /dev/null 2>&1
print("系统依赖装好")

# clone LIBERO
!git clone -q https://github.com/Lifelong-Robot-Learning/LIBERO.git
%cd LIBERO
!pip install -q -e . 2>&1 | tail -5
print("LIBERO 装好")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


系统依赖装好


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


/workspace/LIBERO


/usr/local/lib/python3.11/dist-packages/IPython/core/magics/osm.py:417: UserWarning: This is now an optional IPython functionality, setting dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


  DEPRECATION: Legacy editable install of libero==0.1.0 from file:///workspace/LIBERO (setup.py develop) is deprecated. pip 25.0 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
LIBERO 装好


In [8]:
import os
# 关键:告诉 MuJoCo 用 EGL 离屏渲染(云服务器没显示器)
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"
os.environ["TOKENIZERS_PARALLELISM"] = "false"  # 顺手消掉那个 fork 警告

# 验证 LIBERO 能导入,并列出可用的任务套件
from libero.libero import benchmark
benchmark_dict = benchmark.get_benchmark_dict()
print("可用的任务套件:", list(benchmark_dict.keys()))

Do you want to specify a custom path for the dataset folder? (Y/N):  N


Initializing the default config file...
The following information is stored in the config file: /root/.libero/config.yaml
benchmark_root: /workspace/LIBERO/libero/libero
bddl_files: /workspace/LIBERO/libero/libero/./bddl_files
init_states: /workspace/LIBERO/libero/libero/./init_files
datasets: /workspace/LIBERO/libero/libero/../datasets
assets: /workspace/LIBERO/libero/libero/./assets
可用的任务套件: ['libero_spatial', 'libero_object', 'libero_goal', 'libero_90', 'libero_10', 'libero_100']


In [10]:
!pip install -q robosuite==1.4.1 2>&1 | tail -3
print("robosuite 装好")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
robosuite 装好


In [12]:
!pip install -q bddl==1.0.1 2>&1 | tail -3
print("bddl 装好")


[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
bddl 装好


In [14]:
!pip install -q future 2>&1 | tail -2
print("future 装好")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
future 装好


In [16]:
!pip install -q easydict 2>&1 | tail -2
print("easydict 装好")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
easydict 装好


In [18]:
!pip install -q matplotlib 2>&1 | tail -2
print("matplotlib 装好")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
matplotlib 装好


In [20]:
!pip install -q cloudpickle 2>&1 | tail -2
print("cloudpickle 装好")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
cloudpickle 装好


In [22]:
!pip install -q gym 2>&1 | tail -2
print("gym 装好")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
gym 装好


In [1]:
import os
os.environ["MUJOCO_GL"] = "osmesa"           # 改用软件渲染,绕开 EGL 卡死
os.environ["PYOPENGL_PLATFORM"] = "osmesa"
print("已切换到 osmesa 软件渲染")

已切换到 osmesa 软件渲染


In [2]:
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv
from PIL import Image

benchmark_dict = benchmark.get_benchmark_dict()
task_suite = benchmark_dict["libero_spatial"]()
task = task_suite.get_task(0)
print("任务描述:", task.language)

task_bddl = task_suite.get_task_bddl_file_path(0)
env = OffScreenRenderEnv(bddl_file_name=task_bddl, camera_heights=256, camera_widths=256)
env.seed(0)
env.reset()

obs = env.env._get_observations()
img = obs["agentview_image"][::-1]
Image.fromarray(img).save("first_frame.png")
print("✅ 渲染成功!")
env.close()

[robosuite WARNING] No private macro file found! (macros.py:53)
[robosuite WARNING] It is recommended to use a private macro file (macros.py:54)
[robosuite WARNING] To setup, run: python /usr/local/lib/python3.11/dist-packages/robosuite/scripts/setup_macros.py (macros.py:55)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
任务描述: pick up the black bowl between the plate and the ramekin and place it on the plate
[Warning]: datasets path /workspace/LIBERO/libero/libero/../datasets does not exist!
✅ 渲染成功!


In [3]:
from transformers import AutoModelForVision2Seq, AutoProcessor
import torch

processor = AutoProcessor.from_pretrained("openvla/openvla-7b", trust_remote_code=True)
model = AutoModelForVision2Seq.from_pretrained(
    "openvla/openvla-7b",
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
).to("cuda")
print("模型重新加载完成 ✅")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
<frozen importlib._bootstrap>:283: DeprecationWarning: the load_module() method is deprecated and slated for removal in Python 3.12; use exec_module() instead
/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

/usr/local/lib/python3.11/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


模型重新加载完成 ✅


In [6]:
!pip install -q imageio 2>&1 | tail -2
print("imageio 装好")

[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python -m pip install --upgrade pip
imageio 装好


In [8]:
import os, imageio, numpy as np
from PIL import Image
from libero.libero import benchmark
from libero.libero.envs import OffScreenRenderEnv

task_suite = benchmark.get_benchmark_dict()["libero_spatial"]()
task = task_suite.get_task(0)
instruction = task.language
print("指令:", instruction)

env = OffScreenRenderEnv(
    bddl_file_name=task_suite.get_task_bddl_file_path(0),
    camera_heights=256, camera_widths=256,
)
env.seed(0)
obs = env.reset()

prompt = f"In: What action should the robot take to {instruction}?\nOut:"
frames = []
N = 40

for step in range(N):
    img = obs["agentview_image"][::-1]
    frames.append(img)

    inputs = processor(prompt, Image.fromarray(img))
    inputs = {k: (v.to("cuda", dtype=torch.bfloat16) if v.dtype==torch.float32 else v.to("cuda"))
              for k, v in inputs.items()}
    action = model.predict_action(**inputs, unnorm_key="bridge_orig", do_sample=False)   # ← bridge_orig

    obs, reward, done, info = env.step(action.tolist())
    if step % 10 == 0:
        print(f"step {step}: action={np.round(action,3)}")
    if done:
        print("done!"); break

env.close()
imageio.mimsave("rollout.gif", frames, fps=10)
print(f"✅ 闭环跑完 {len(frames)} 步,动图存为 rollout.gif")

[info] using task orders [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
指令: pick up the black bowl between the plate and the ramekin and place it on the plate
[Warning]: datasets path /workspace/LIBERO/libero/libero/../datasets does not exist!
step 0: action=[-0.001 -0.    -0.004  0.008 -0.017 -0.003  0.996]
step 10: action=[-0.009 -0.002 -0.001  0.016 -0.005 -0.01   0.996]
step 20: action=[ 0.003 -0.019  0.003 -0.022  0.005 -0.095  0.996]
step 30: action=[ 0.    -0.     0.     0.005 -0.003 -0.018  0.996]


/usr/local/lib/python3.11/dist-packages/imageio/plugins/pillow.py:410: DeprecationWarning: The keyword `fps` is no longer supported. Use `duration`(in ms) instead, e.g. `fps=50` == `duration=20` (1000 * 1/50).
  warnings.warn(


✅ 闭环跑完 40 步,动图存为 rollout.gif
